In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CustomerDtProcessing").getOrCreate()
spark


In [0]:
dfr = spark.read.format("csv").option("header",  "true").load("/Workspace/Users/bhagatishaanraj43@gmail.com/customers.csv")

In [0]:
dfr.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows


In [0]:
dfr.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: string (nullable = true)



In [0]:
from pyspark.sql.functions import *
#changing date to date-time from string, is_active to boolean from string
dfr = dfr.withColumn('registration_date', to_date(col("registration_date"), 'yyyy-MM-dd')) \
      .withColumn('is_active', col("is_active").cast("boolean"))

In [0]:
dfr = dfr.fillna({'city':'Unknown', 'state':'Unknown', 'country':'Unknown'})

In [0]:
dfr = dfr.withColumn('reg_yr', year(col('registration_date'))) \
     .withColumn('reg_month', month(col('registration_date')))

In [0]:
dfr.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|reg_yr|reg_month|
+-----------+----------+---------+-----------+-------+-----------------+---------+------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|  2023|        6|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|  2023|       12|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|  2023|       10|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|  2023|       10|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|  2023|        3|
+-----------+----------+---------+-----------+-------+-----------------+---------+------+---------+
only showing top 5 rows


In [0]:
dfr.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- country: string (nullable = false)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- reg_yr: integer (nullable = true)
 |-- reg_month: integer (nullable = true)



In [0]:
dfr.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|reg_yr|reg_month|
+-----------+----------+---------+-----------+-------+-----------------+---------+------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|  2023|        6|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|  2023|       12|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|  2023|       10|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|  2023|       10|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|  2023|        3|
+-----------+----------+---------+-----------+-------+-----------------+---------+------+---------+
only showing top 5 rows


In [0]:
unique_cities = dfr.select(countDistinct('city')).collect()
print(f'Num of unique cities:{unique_cities[0][0]}')
unique_states = dfr.select(countDistinct('state')).collect()
print(f'Num of unique states:{unique_states[0][0]}')
unique_countries = dfr.select(countDistinct('country')).collect()
print(f'Num of unique countries:{unique_countries[0][0]}')

Num of unique cities:8
Num of unique states:7
Num of unique countries:1


In [0]:
dfr.groupBy('city').count().orderBy(col('count').desc()).show()

+---------+-----+
|     city|count|
+---------+-----+
|     Pune| 2243|
|Hyderabad| 2242|
|  Kolkata| 2223|
|Bangalore| 2211|
|    Delhi| 2200|
|Ahmedabad| 2198|
|  Chennai| 2194|
|   Mumbai| 2142|
+---------+-----+



In [0]:
dfr.groupBy('state').count().orderBy(col('count').desc()).show()

+-----------+-----+
|      state|count|
+-----------+-----+
|      Delhi| 2578|
|    Gujarat| 2543|
| Tamil Nadu| 2536|
|  Telangana| 2520|
|West Bengal| 2503|
|Maharashtra| 2490|
|  Karnataka| 2483|
+-----------+-----+



In [0]:
#active inactive users count
dfr.groupBy('state').pivot('is_active').count().show()

+-----------+-----+----+
|      state|false|true|
+-----------+-----+----+
|      Delhi| 1356|1222|
|  Telangana| 1294|1226|
| Tamil Nadu| 1284|1252|
|    Gujarat| 1211|1332|
|West Bengal| 1306|1197|
|Maharashtra| 1260|1230|
|  Karnataka| 1207|1276|
+-----------+-----+----+



####Applying window function

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy('state').orderBy(col('registration_date').desc())

dfr = dfr.withColumn('rank', rank().over(window_spec))\
    .withColumn('dense_rank', dense_rank().over(window_spec))\
        .withColumn('row_number', row_number().over(window_spec))


In [0]:
dfr.select('name', 'city', 'state', 'rank', 'dense_rank', 'row_number').show(5)

+--------------+---------+-----+----+----------+----------+
|          name|     city|state|rank|dense_rank|row_number|
+--------------+---------+-----+----+----------+----------+
|   Customer_61|Hyderabad|Delhi|   1|         1|         1|
|  Customer_501|   Mumbai|Delhi|   1|         1|         2|
| Customer_2763|     Pune|Delhi|   1|         1|         3|
|Customer_12858|Ahmedabad|Delhi|   1|         1|         4|
|Customer_13570|Bangalore|Delhi|   1|         1|         5|
+--------------+---------+-----+----+----------+----------+
only showing top 5 rows


####getting latest data of recent customers

In [0]:
dfr_recent_cust = dfr.filter(col("registration_date") >= lit('2023-07-01'))
dfr_recent_cust.show()

+-----------+--------------+---------+-----+-------+-----------------+---------+------+---------+----+----------+----------+
|customer_id|          name|     city|state|country|registration_date|is_active|reg_yr|reg_month|rank|dense_rank|row_number|
+-----------+--------------+---------+-----+-------+-----------------+---------+------+---------+----+----------+----------+
|         61|   Customer_61|Hyderabad|Delhi|  India|       2023-12-31|    false|  2023|       12|   1|         1|         1|
|        501|  Customer_501|   Mumbai|Delhi|  India|       2023-12-31|    false|  2023|       12|   1|         1|         2|
|       2763| Customer_2763|     Pune|Delhi|  India|       2023-12-31|     true|  2023|       12|   1|         1|         3|
|      12858|Customer_12858|Ahmedabad|Delhi|  India|       2023-12-31|     true|  2023|       12|   1|         1|         4|
|      13570|Customer_13570|Bangalore|Delhi|  India|       2023-12-31|    false|  2023|       12|   1|         1|         5|


In [0]:
dfr.count()

17653

In [0]:
dfr_recent_cust.count()

9025

####oldest and the newest customer per city

In [0]:
dfr.groupBy('city').agg(min('registration_date').alias('oldest'), max('registration_date').alias('newest')).show()

+---------+----------+----------+
|     city|    oldest|    newest|
+---------+----------+----------+
|    Delhi|2023-01-01|2023-12-31|
|  Chennai|2023-01-01|2023-12-31|
|Ahmedabad|2023-01-01|2023-12-31|
|  Kolkata|2023-01-01|2023-12-31|
|Hyderabad|2023-01-01|2023-12-31|
|     Pune|2023-01-01|2023-12-31|
|Bangalore|2023-01-01|2023-12-31|
|   Mumbai|2023-01-01|2023-12-31|
+---------+----------+----------+



In [0]:
out_path = '/Volumes/workspace/default/processed_data_vol/customer_out.parquet'
dfr.write.mode('overwrite').parquet(out_path)

###Joining and Analyzing Customers and Orders

In [0]:
ordr_dfr = spark.read.format('csv').option("header", "true").option('inferSchema', True).load("/Workspace/Users/bhagatishaanraj43@gmail.com/orders.csv")

In [0]:
ordr_dfr.show(5)

+--------+-----------+----------+-----------------+---------+
|order_id|customer_id|order_date|     total_amount|   status|
+--------+-----------+----------+-----------------+---------+
|       0|       3692|2024-09-03|547.7160076008001|  Shipped|
|       1|      11055|2024-08-10|577.8942599188381|  Pending|
|       2|       6963|2024-08-22|484.2085562764487|  Pending|
|       3|      13268|2024-09-01|366.3286882431848|Cancelled|
|       4|       1131|2024-08-09|896.9588380686909|  Pending|
+--------+-----------+----------+-----------------+---------+
only showing top 5 rows


In [0]:
ordr_dfr.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- status: string (nullable = true)



In [0]:
customer_orders_dfr = dfr.join(ordr_dfr, 'customer_id', "inner")

In [0]:
customer_orders_dfr.count()

17653

In [0]:
customer_orders_dfr.show(5)

+-----------+--------------+---------+-----+-------+-----------------+---------+------+---------+----+----------+----------+--------+----------+------------------+---------+
|customer_id|          name|     city|state|country|registration_date|is_active|reg_yr|reg_month|rank|dense_rank|row_number|order_id|order_date|      total_amount|   status|
+-----------+--------------+---------+-----+-------+-----------------+---------+------+---------+----+----------+----------+--------+----------+------------------+---------+
|         61|   Customer_61|Hyderabad|Delhi|  India|       2023-12-31|    false|  2023|       12|   1|         1|         1|   17283|2024-01-21|171.91694722937515|  Pending|
|        501|  Customer_501|   Mumbai|Delhi|  India|       2023-12-31|    false|  2023|       12|   1|         1|         2|   16373|2024-10-29| 944.5362648102838|  Pending|
|       2763| Customer_2763|     Pune|Delhi|  India|       2023-12-31|     true|  2023|       12|   1|         1|         3|   137

In [0]:
customer_orders_dfr.display(5)

In [0]:
#Total Orders per customer

customers_ordr_count = customer_orders_dfr.groupBy('customer_id').count().orderBy(col('count').desc())
customers_ordr_count.show(10)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|      11776|    7|
|       4294|    6|
|       7566|    6|
|       5160|    6|
|       3243|    6|
|      14838|    6|
|      13034|    6|
|       3884|    6|
|       3336|    6|
|      11537|    5|
+-----------+-----+
only showing top 10 rows


In [0]:
#total money spend by per consumer

custmr_total_spend = customer_orders_dfr.groupBy('customer_id').agg(sum('total_amount')).orderBy(col('sum(total_amount)').desc())
custmr_total_spend.show(10)

+-----------+------------------+
|customer_id| sum(total_amount)|
+-----------+------------------+
|       3336| 4362.550733141537|
|       3884|  4187.99763145619|
|      16020|3967.2692112582276|
|      14372| 3961.787139557334|
|      14933|3828.5841072418348|
|       7566| 3647.119115720654|
|      10559|3548.8378633460234|
|      11776|  3438.36692751212|
|      11449| 3396.060974816134|
|       5425| 3389.162933156913|
+-----------+------------------+
only showing top 10 rows


#####from above two cells, customer id 3336, 3884 are concluded to be the premium customers.

In [0]:
#Average spend per customer

custmr_avg_spend = customer_orders_dfr.groupBy('customer_id').agg(avg('total_amount')).orderBy(col('avg(total_amount)').desc())
custmr_avg_spend.show(10)

+-----------+-----------------+
|customer_id|avg(total_amount)|
+-----------+-----------------+
|      11854| 999.864258397557|
|         46| 999.592553819927|
|      17590|999.5726342625253|
|      11587|999.5595016039513|
|       6816|999.4348902885968|
|       9648| 999.421469440536|
|      15486|999.0348855189945|
|      10980|998.9071094160754|
|       1711|998.7736900580057|
|       6333|998.6394502434505|
+-----------+-----------------+
only showing top 10 rows


In [0]:
#order by status

ordr_status_count = customer_orders_dfr.groupBy('status').count()
ordr_status_count.show()

+---------+-----+
|   status|count|
+---------+-----+
|  Pending| 4457|
|  Shipped| 4386|
|Delivered| 4341|
|Cancelled| 4469|
+---------+-----+



In [0]:
#Orders by month

ordr_by_month = customer_orders_dfr.withColumn('order_month', month(col('order_date')))\
    .groupBy('order_month').count()\
        .orderBy('order_month')

ordr_by_month.show()

+-----------+-----+
|order_month|count|
+-----------+-----+
|          1| 1499|
|          2| 1368|
|          3| 1539|
|          4| 1457|
|          5| 1518|
|          6| 1455|
|          7| 1517|
|          8| 1472|
|          9| 1446|
|         10| 1513|
|         11| 1426|
|         12| 1443|
+-----------+-----+



In [0]:
# Rank the customer by total spend usingn window func

window_spc = Window.orderBy(col('sum(total_amount)').desc())

rankd_custmr = custmr_total_spend.withColumn('dense_rank', dense_rank().over(window_spc))
rankd_custmr.show(10)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------------------+----------+
|customer_id| sum(total_amount)|dense_rank|
+-----------+------------------+----------+
|       3336| 4362.550733141537|         1|
|       3884|  4187.99763145619|         2|
|      16020|3967.2692112582276|         3|
|      14372| 3961.787139557334|         4|
|      14933|3828.5841072418348|         5|
|       7566| 3647.119115720654|         6|
|      10559|3548.8378633460234|         7|
|      11776|  3438.36692751212|         8|
|      11449| 3396.060974816134|         9|
|       5425| 3389.162933156913|        10|
+-----------+------------------+----------+
only showing top 10 rows


In [0]:
custmr_total_spend.printSchema(), customers_ordr_count.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- sum(total_amount): double (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- count: long (nullable = false)



(None, None)

In [0]:
#Finding customers with High order frequency but low total spend

custmr_spend_vs_ordr = customers_ordr_count.join(custmr_total_spend, 'customer_id','inner')\
    .orderBy(col('count').desc(), col('sum(total_amount)'))

custmr_spend_vs_ordr.show(5)

+-----------+-----+------------------+
|customer_id|count| sum(total_amount)|
+-----------+-----+------------------+
|      11776|    7|  3438.36692751212|
|       5160|    6| 1656.737343311546|
|       4294|    6| 1821.603928366352|
|       3243|    6|2860.1827303387754|
|      14838|    6| 2894.355602564058|
+-----------+-----+------------------+
only showing top 5 rows


In [0]:
out_pth = '/Volumes/workspace/default/processed_data_vol/customer_and_orders_out.parquet'
customer_orders_dfr.write.mode('overwrite').parquet(out_pth)

In [0]:
pandas_df = customer_orders_dfr.toPandas()

out_csv_path = "/Workspace/Users/bhagatishaanraj43@gmail.com/Mini_ecommerce_project/customer_and_orders_output.csv"
pandas_df.to_csv(out_csv_path, index=False)